In [4]:
import numpy as np
import plotly.graph_objects as go
import torch

In [3]:
# 1D Dataset Generation with Outliers
def generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.03):
    # Generate x values uniformly distributed between 0 and 2π
    x = np.linspace(0, 2 * np.pi, n)
    
    # True function: sine wave
    y_true = np.sin(2 * x)
    
    # Add Gaussian noise to y values
    y = y_true + np.random.normal(0, noise_std, x.shape[0])
    
    # Insert outliers
    n_outliers = int(n * outlier_ratio)
    outlier_indices = np.random.choice(n, n_outliers, replace=False)
    y[outlier_indices] = 5 + np.random.uniform(-0.1, 0.1, n_outliers)
    
    return x, y, y_true, outlier_indices


# Generate 1D dataset with outliers
x, y, y_true, outlier_indices = generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05)

# Convert the data to numpy arrays for Plotly
x = x
y = y
y_true = y_true
outliers_x = x[outlier_indices]
outliers_y = y[outlier_indices]

# Create traces for plotly
true_func_trace = go.Scatter(x=x, y=y_true, mode='lines', name='True function (sin(2x))', line=dict(color='red'))
data_trace = go.Scatter(x=x, y=y, mode='markers', name='Data with noise and outliers', marker=dict(color='blue', size=6))
outliers_trace = go.Scatter(x=outliers_x, y=outliers_y, mode='markers', name='Outliers', marker=dict(color='orange', size=8))

# Combine the traces
fig = go.Figure(data=[true_func_trace, data_trace, outliers_trace])

# Update layout for better visualization
fig.update_layout(
    title="1D Data with Noise and Outliers",
    xaxis_title="x",
    yaxis_title="y",
    showlegend=True,
    legend=dict(
        x=0,
        y=-0.2,
        orientation="h"
    )
)

# Show plot
fig.show()


In [7]:
from _src import EvidentialMLP

num_epochs = 50
input_dim = 1
output_dim = 1


X, y, y_true, outlier_indices, X_test = generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05)

X_tensor = torch.tensor(X, dtype=torch.float32).reshape(-1, 1)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

model = EvidentialMLP(input_dim=1, output_dim=1, num_models=3)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(num_epochs):
    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = model.compute_loss(preds, y_tensor, X_tensor)
    loss.backward()
    optimizer.step()



In [6]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

NameError: name 'X' is not defined

In [ ]:
# モデルの訓練
def train_model(X, y, model, optimizer, hovr_lambda=None, hovr_k=None, hovr_q=None, num_epochs=5000):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
    
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = nn.MSELoss()(preds, y_tensor)
        
        # HOVR正則化を追加
        if hovr_lambda is not None and hovr_k is not None and hovr_q is not None:
            hovr_term = hovr_regularization(model, X_tensor, k=hovr_k, q=hovr_q)
            loss += hovr_lambda * hovr_term

        loss.backward()
        optimizer.step()

In [ ]:
if __name__ == "__main__":
    # Generate toy dataset
    x_train, y_train, x_test, y_test = generate_toy_data()

    # Initialize model with customized settings
    model_config = {
        "input_dim": 1,
        "output_dim": 1,
        "hidden_units": [128, 128, 128],
        "activations": ["tanh", "relu", "relu"],
        "nig_coeff": 1e-2,
        # "nsu_coeff": 1e-3,
        "nsu_coeff": 0,
        "hovr_coeff": 1e-4,
        # "hovr_coeff": 0,
    }
    model = EvidentialMLP(**model_config)

    # Train the model
    train_model(model, x_train, y_train)

    # Plot the results
    plot_results(model, x_train, y_train, x_test, y_test)

In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go

# Assuming the EvidentialMLP and its related functions are defined as per your earlier code

# 1D Dataset Generation with Outliers
def generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05):
    # Generate x values uniformly distributed between 0 and 2π
    x = np.linspace(0, 2 * np.pi, n)
    
    # True function: sine wave
    y_true = np.sin(2 * x)
    
    # Add Gaussian noise to y values
    y = y_true + np.random.normal(0, noise_std, x.shape[0])
    
    # Insert outliers
    n_outliers = int(n * outlier_ratio)
    outlier_indices = np.random.choice(n, n_outliers, replace=False)
    y[outlier_indices] = 5 + np.random.uniform(-0.1, 0.1, n_outliers)
    
    return x, y, y_true, outlier_indices

# Generate 1D dataset with outliers
x, y, y_true, outlier_indices = generate_1d_data(n=200, noise_std=0.2, outlier_ratio=0.05)

# Convert data to tensors
X_tensor = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

# Initialize model with customized settings
model_config = {
    "input_dim": 1,
    "output_dim": 1,
    "hidden_units": [128, 128, 128],
    "activations": ["tanh", "relu", "relu"],
    "nig_coeff": 1e-2,
    # "nsu_coeff": 1e-3,
    "nsu_coeff": 0,
    "hovr_coeff": 1e-4,
    # "hovr_coeff": 0,
}
model = EvidentialMLP(**model_config)

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Full-batch training loop
num_epochs = 500

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # Full batch passed to model
    preds = model(X_tensor)
    
    # Compute the loss
    loss = model.compute_loss(preds, y_tensor, X_tensor)
    
    # Backpropagation and optimization
    loss.backward()
    optimizer.step()

# Generate test data for plotting
x_test = np.linspace(0, 2 * np.pi, 1000)
X_test_tensor = torch.tensor(x_test, dtype=torch.float32).reshape(-1, 1)

# Get model predictions
model.eval()
with torch.no_grad():
    pred = model(X_test_tensor)
    mu, std = model.predict(X_test_tensor, pred)
    
# Convert to numpy for plotting
x_test_np = x_test
mu_np = mu.numpy().squeeze()
std_np = std.numpy().squeeze()
y_true_np = np.sin(2 * x_test)

# Create traces for plotly
true_func_trace = go.Scatter(
    x=x_test_np, y=y_true_np, mode='lines', name='True function (sin(2x))', line=dict(color='red')
)
pred_trace = go.Scatter(
    x=x_test_np, y=mu_np, mode='lines', name='Predicted Mean', line=dict(color='blue', dash='dash')
)
upper_bound_trace = go.Scatter(
    x=x_test_np,
    y=mu_np + 2 * std_np,
    fill=None,
    mode='lines',
    line=dict(color='lightblue'),
    name='Upper Bound (Mean + 2*Std)',
    showlegend=False
)
lower_bound_trace = go.Scatter(
    x=x_test_np,
    y=mu_np - 2 * std_np,
    fill='tonexty',
    mode='lines',
    line=dict(color='lightblue'),
    name='Lower Bound (Mean - 2*Std)',
    showlegend=False
)
data_trace = go.Scatter(
    x=x, y=y, mode='markers', name='Training Data', marker=dict(color='black', size=5)
)
outliers_trace = go.Scatter(
    x=x[outlier_indices], y=y[outlier_indices], mode='markers', name='Outliers', marker=dict(color='orange', size=8)
)

# Combine all traces
fig = go.Figure(data=[true_func_trace, pred_trace, upper_bound_trace, lower_bound_trace, data_trace, outliers_trace])

# Update layout for better visualization
fig.update_layout(
    title="1D Regression with Outliers using EvidentialMLP (Full Batch Training)",
    xaxis_title="x",
    yaxis_title="y",
    showlegend=True,
    legend=dict(
        x=0,
        y=-0.2,
        orientation="h"
    )
)

# Show plot
fig.show()


In [3]:
import plotly.graph_objects as go

# 1D Dataset Generation with Outliers
def generate_1d_data(n=100, noise_std=0.2, outlier_ratio=0.05):
    # Generate x values uniformly distributed between 0 and 2π
    x = np.linspace(0, 2 * np.pi, n)
    
    # True function: sine wave
    y_true = np.sin(2 * x)
    
    # Add Gaussian noise to y values
    y = y_true + np.random.normal(0, noise_std, x.shape[0])
    
    # Insert outliers
    n_outliers = int(n * outlier_ratio)
    outlier_indices = np.random.choice(n, n_outliers, replace=False)
    y[outlier_indices] = 5 + np.random.uniform(-0.1, 0.1, n_outliers)
    
    return x, y, y_true, outlier_indices


# Plot function for results
def plot_results(model, x_test, y_true, x, y, outlier_indices):
    # Generate test data
    X_test_tensor = torch.tensor(x_test, dtype=torch.float32).reshape(-1, 1)
    
    # Get model predictions
    model.eval()
    with torch.no_grad():
        pred = model(X_test_tensor)
        mu, std = model.predict(X_test_tensor, pred)
        
    # Convert to numpy for plotting
    x_test_np = x_test
    mu_np = mu.numpy().squeeze()
    std_np = std.numpy().squeeze()
    y_true_np = y_true

    # Create traces for plotly
    true_func_trace = go.Scatter(
        x=x_test_np, y=y_true_np, mode='lines', name='True function (sin(2x))', line=dict(color='red')
    )
    pred_trace = go.Scatter(
        x=x_test_np, y=mu_np, mode='lines', name='Predicted Mean', line=dict(color='blue', dash='dash')
    )
    upper_bound_trace = go.Scatter(
        x=x_test_np,
        y=mu_np + 2 * std_np,
        fill=None,
        mode='lines',
        line=dict(color='lightblue'),
        name='Upper Bound (Mean + 2*Std)',
        showlegend=False
    )
    lower_bound_trace = go.Scatter(
        x=x_test_np,
        y=mu_np - 2 * std_np,
        fill='tonexty',
        mode='lines',
        line=dict(color='lightblue'),
        name='Lower Bound (Mean - 2*Std)',
        showlegend=False
    )
    data_trace = go.Scatter(
        x=x, y=y, mode='markers', name='Training Data', marker=dict(color='black', size=5)
    )
    outliers_trace = go.Scatter(
        x=x[outlier_indices], y=y[outlier_indices], mode='markers', name='Outliers', marker=dict(color='orange', size=8)
    )

    # Combine all traces
    fig = go.Figure(data=[true_func_trace, pred_trace, upper_bound_trace, lower_bound_trace, data_trace, outliers_trace])

    # Update layout for better visualization
    fig.update_layout(
        title="1D Regression with Outliers using EvidentialMLP (Full Batch Training)",
        xaxis_title="x",
        yaxis_title="y",
        showlegend=True,
        legend=dict(
            x=0,
            y=-0.2,
            orientation="h"
        )
    )

    # Show plot
    fig.show()

In [31]:
import numpy as np
import torch

from _src import EvidentialMLP, train_ern

# Generate 1D dataset with outliers
x, y, y_true, outlier_indices = generate_1d_data(n=200, noise_std=0.2, outlier_ratio=0.05)
    
# Convert data to tensors
X_tensor = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
    
# Initialize model with customized settings
model_config = {
    "input_dim": 1,
    "output_dim": 1,
    "hidden_units": [128, 128, 128],
    "activations": ["tanh", "tanh", "tanh"],
    "nig_coeff": 1e-2,
    "nsu_coeff": 1e-3,
    # "hovr_coeff": 1e-4,
    "hovr_coeff": 0,
}
model = EvidentialMLP(**model_config)
    
# Train the model using full-batch gradient descent
train_ern(model, X_tensor, y_tensor, num_epochs=10000, lr=0.01)
    
# Generate test data for plotting
x_test = np.linspace(0, 2 * np.pi, 1000)
y_true_test = np.sin(2 * x_test)
    
# Plot the results
plot_results(model, x_test, y_true_test, x, y, outlier_indices)